# 📊 skfolio 빠른 시작: 포트폴리오 최적화 기초

이 노트북은 **`skfolio`**를 활용하여 자산 배분 모델(평균-분산, 최소 분산 등)을 학습하고 최적의 투자 비중을 도출하는 튜토리얼입니다.

## 1. 환경 준비 및 라이브러리 임포트

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트 및 src 디렉토리를 sys.path에 추가하여 skfolio 로컬 모듈 탐색 보장
project_root = str(Path.cwd().parent.resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
src_dir = str(Path(project_root) / 'src')
for p in [project_root, src_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from skfolio import RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.optimization import MeanVariance, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns

print("라이브러리가 성공적으로 로드되었습니다.")

## 2. 데이터셋 로드 및 수익률 변환

`skfolio`의 내장 예제 데이터셋(주요 자산 5개)을 불러옵니다.

In [ ]:
# 예제 데이터셋 로드 (S&P 500 내 주요 종목)
prices = load_sp500_dataset()
selected_assets = ["AAPL", "MSFT", "AMZN", "JNJ", "JPM"]
prices = prices[selected_assets]

# 가격 데이터를 수익률(Returns)로 변환
returns = prices_to_returns(prices)
returns.tail()

## 3. 포트폴리오 최적화 모델 학습

1. **최대 샤프 지수 (Maximum Sharpe Ratio)**: 위험 대비 수익률을 극대화
2. **최소 분산 (Minimum Variance)**: 전체 포트폴리오의 변동성을 극소화

In [ ]:
# 1) 최대 샤프 지수 모델
model_sharpe = MeanVariance(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
)
model_sharpe.fit(returns)

# 2) 최소 분산 모델
model_min_var = MeanVariance(
    objective_function=ObjectiveFunction.MINIMIZE_RISK,
    risk_measure=RiskMeasure.VARIANCE,
)
model_min_var.fit(returns)

weights_df = pd.DataFrame({
    "Max Sharpe": model_sharpe.weights_,
    "Min Variance": model_min_var.weights_,
}, index=selected_assets)

(weights_df * 100).round(2).astype(str) + "%"

## 4. 포트폴리오 성과 평가 및 비교

도출된 각 포트폴리오의 기대 수익률, 연간 변동성, 샤프 지수를 비교합니다.

In [ ]:
port_sharpe = model_sharpe.predict(returns)
port_min_var = model_min_var.predict(returns)

summary = pd.DataFrame({
    "Max Sharpe": [
        f"{port_sharpe.mean * 252 * 100:.2f}%",
        f"{port_sharpe.variance**0.5 * (252**0.5) * 100:.2f}%",
        f"{port_sharpe.mean / (port_sharpe.variance**0.5 + 1e-9) * (252**0.5):.2f}",
    ],
    "Min Variance": [
        f"{port_min_var.mean * 252 * 100:.2f}%",
        f"{port_min_var.variance**0.5 * (252**0.5) * 100:.2f}%",
        f"{port_min_var.mean / (port_min_var.variance**0.5 + 1e-9) * (252**0.5):.2f}",
    ],
}, index=["연환산 기대수익률", "연환산 변동성", "샤프 지수 (Sharpe Ratio)"])

summary